# Análise de Negócio — Detecção de Fraudes

**Objetivo:** Avaliar se o modelo de detecção de fraudes compensa financeiramente, quantificando o tradeoff entre fraudes capturadas e transações legítimas negadas.

**Perguntas respondidas:**
1. Qual o impacto financeiro do modelo vs não ter modelo?
2. O tradeoff (fraudes pegas × falsos positivos) é favorável?
3. Qual threshold maximiza o retorno financeiro líquido?
4. Qual a exposição residual (fraudes que ainda passam)?
5. O impacto é estável ao longo do tempo (por safra)?
6. Quão sensível é o resultado a diferentes premissas de custo?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import glob
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('husl')

RANDOM_STATE = 42

---
## 1. Parâmetros de Negócio

Defina as premissas financeiras abaixo. Todos os valores em R$.

In [ ]:
# ── Premissas financeiras (ajuste conforme a realidade do negócio) ──────

# Custo de uma transação legítima negada (Falso Positivo)
CUSTO_REVISAO_FP = 5.00    # R$ — custo fixo de revisar/bloquear uma transação legítima
TAXA_CHURN_FP    = 0.02    # 2% dos clientes com FP cancelam o cartão
LTV_CLIENTE      = 800.00  # R$ — valor médio anual de um cliente retido

# Custo total de um FP = custo operacional + perda esperada de churn
CUSTO_FP = CUSTO_REVISAO_FP + (TAXA_CHURN_FP * LTV_CLIENTE)

# Custo operacional de manter o modelo — por período de análise
CUSTO_OPERACIONAL_MODELO = 0.00  # R$ — inclua se quiser no ROI

print('Premissas de Custo:')
print(f'  Custo por FP (transação legítima negada) : R$ {CUSTO_FP:.2f}')
print(f'    └─ Revisão operacional                 : R$ {CUSTO_REVISAO_FP:.2f}')
print(f'    └─ Perda esperada por churn            : R$ {TAXA_CHURN_FP*LTV_CLIENTE:.2f}  ({TAXA_CHURN_FP:.0%} x R${LTV_CLIENTE:.0f})')
print(f'  Custo por FN (fraude não detectada)      : R$ TX_AMOUNT (variável por transação)')

---
## 2. Carregamento do Modelo e Dados

In [ ]:
# ── Modelo mais recente (XGBoost Optuna) ────────────────────────────────
model_files = sorted(glob.glob('../data/models/xgboost_optuna*.pkl'))
assert model_files, 'Nenhum modelo encontrado em data/models/. Execute o ExperimentLoggerAgent primeiro.'
MODEL_PATH = model_files[-1]
print(f'Modelo carregado: {os.path.basename(MODEL_PATH)}')

pipeline = joblib.load(MODEL_PATH)

# Extrair threshold do nome do arquivo via regex
# Suporta formatos: thr0.300_<runid>.pkl  e  thr0_300_<runid>.pkl
_match = re.search(r'thr([\d._]+)_[0-9a-f]{8}', os.path.basename(MODEL_PATH))
if _match:
    THRESH_OPT = float(_match.group(1).rstrip('_').replace('_', '.'))
else:
    THRESH_OPT = 0.30
print(f'Threshold de classificação: {THRESH_OPT:.3f}')

In [ ]:
# ── Features e split (idêntico ao notebook de modelagem) ─────────────────
X = pd.read_parquet('../data/gold/X_train.parquet')
y = pd.read_parquet('../data/gold/y_train.parquet').squeeze()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# TX_AMOUNT do gold — não entra no modelo, mas é essencial para o custo financeiro
gold = pd.read_parquet('../data/gold/train_gold.parquet', columns=['TX_AMOUNT'])
_, gold_val, _, _ = train_test_split(
    gold, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
amount_val = gold_val['TX_AMOUNT'].values

# Safra (mês de referência) a partir do silver
silver = pd.read_parquet('../data/silver/train_silver.parquet', columns=['TX_DATETIME'])
_, silver_val, _, _ = train_test_split(
    silver, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
safra_val = silver_val['TX_DATETIME'].dt.to_period('M').astype(str).values

print(f'Validação : {X_val.shape[0]:,} transações | fraude: {y_val.mean():.4%}')
print(f'Montante total de fraudes (val): R$ {amount_val[y_val.values == 1].sum():,.2f}')

---
## 3. Predições no Conjunto de Validação

In [ ]:
prob_val = pipeline.predict_proba(X_val)[:, 1]
pred_val = (prob_val >= THRESH_OPT).astype(int)
y_arr    = y_val.values

tn, fp, fn, tp = confusion_matrix(y_arr, pred_val).ravel()

print(f'Threshold : {THRESH_OPT:.3f}')
print()
print(f'  TP (fraudes detectadas)       : {tp:>6,}')
print(f'  FP (legítimas negadas)        : {fp:>6,}')
print(f'  TN (legítimas aprovadas)      : {tn:>6,}')
print(f'  FN (fraudes não detectadas)   : {fn:>6,}')
print()
print(f'  Recall de fraude (TPR)        : {tp/(tp+fn):.2%}')
print(f'  Taxa de falsos positivos (FPR): {fp/(fp+tn):.4%}')
print(f'  Precisão                      : {tp/(tp+fp):.2%}')

---
## 4. Matriz de Custo-Benefício

In [ ]:
tp_mask = (pred_val == 1) & (y_arr == 1)
fp_mask = (pred_val == 1) & (y_arr == 0)
fn_mask = (pred_val == 0) & (y_arr == 1)

valor_salvo_tp   = amount_val[tp_mask].sum()    # fraudes bloqueadas (R$ salvo)
custo_fp_total   = fp * CUSTO_FP                # custo das negações indevidas
prejuizo_fn      = amount_val[fn_mask].sum()    # fraudes que passaram
perda_sem_modelo = amount_val[y_arr == 1].sum() # baseline: 100% das fraudes passam
beneficio_liquido = valor_salvo_tp - custo_fp_total - CUSTO_OPERACIONAL_MODELO

print('=' * 56)
print(' MATRIZ FINANCEIRA — CONJUNTO DE VALIDAÇÃO')
print('=' * 56)
print(f'  {"Fraudes bloqueadas (TP)":<35}: R$ {valor_salvo_tp:>10,.2f}')
print(f'  {"Custo transações negadas (FP)":<35}: R$ {custo_fp_total:>10,.2f}')
print(f'  {"Fraudes não detectadas (FN)":<35}: R$ {prejuizo_fn:>10,.2f}')
print('-' * 56)
print(f'  {"Benefício líquido c/ modelo":<35}: R$ {beneficio_liquido:>10,.2f}')
print(f'  {"Exposição sem modelo":<35}: R$ {perda_sem_modelo:>10,.2f}')
print(f'  {"Redução de exposição":<35}: {valor_salvo_tp/perda_sem_modelo:>10.2%}')
print('=' * 56)
print()
print(f'  Razão TP/FP : {tp}/{fp} → {tp/max(fp,1):.1f} fraudes bloqueadas por cada legítima negada')
print(f'  Custo médio FP vs valor médio de fraude : R${CUSTO_FP:.2f} vs R${amount_val[y_arr==1].mean():.2f}')

---
## 5. Impacto Financeiro por Threshold

Varremos todos os thresholds para identificar o ponto de máximo benefício líquido. O threshold ótimo financeiro pode diferir do threshold ótimo por F2.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 300)
resultados = []

for thr in thresholds:
    p = (prob_val >= thr).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_arr, p).ravel()
    salvo_  = amount_val[(p == 1) & (y_arr == 1)].sum()
    custo_  = fp_ * CUSTO_FP
    resid_  = amount_val[(p == 0) & (y_arr == 1)].sum()
    resultados.append({
        'threshold'   : thr,
        'tp'          : tp_,
        'fp'          : fp_,
        'fn'          : fn_,
        'valor_salvo' : salvo_,
        'custo_fp'    : custo_,
        'prejuizo_fn' : resid_,
        'beneficio_liq': salvo_ - custo_,
        'recall'      : tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0,
        'precisao'    : tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0,
    })

df_thr = pd.DataFrame(resultados)

idx_fin    = df_thr['beneficio_liq'].idxmax()
THRESH_FIN = df_thr.loc[idx_fin, 'threshold']
BEN_FIN    = df_thr.loc[idx_fin, 'beneficio_liq']

idx_f2  = (df_thr['threshold'] - THRESH_OPT).abs().idxmin()
BEN_F2  = df_thr.loc[idx_f2, 'beneficio_liq']

print(f'Threshold otimizado (F2)     : {THRESH_OPT:.3f}  → benefício líq. R$ {BEN_F2:,.2f}')
print(f'Threshold ótimo financeiro   : {THRESH_FIN:.3f}  → benefício líq. R$ {BEN_FIN:,.2f}')
print(f'Ganho adicional potencial    : R$ {BEN_FIN - BEN_F2:,.2f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

# Painel 1: impacto financeiro
ax1 = axes[0]
ax1.plot(df_thr['threshold'], df_thr['beneficio_liq'],  color='#2196F3', lw=2.5, label='Benefício Líquido (Salvo - Custo FP)')
ax1.plot(df_thr['threshold'], df_thr['valor_salvo'],    color='#4CAF50', lw=1.5, ls='--', label='Fraudes Bloqueadas (R$)')
ax1.plot(df_thr['threshold'], -df_thr['custo_fp'],      color='#F44336', lw=1.5, ls='--', label='Custo FP (negativo)')
ax1.plot(df_thr['threshold'], -df_thr['prejuizo_fn'],   color='#FF9800', lw=1.5, ls=':',  label='Exposição Residual FN (negativo)')
ax1.axvline(THRESH_OPT, color='gray',    ls='--', lw=1.2, label=f'Threshold F2 ({THRESH_OPT:.3f})')
ax1.axvline(THRESH_FIN, color='#1565C0', ls='-',  lw=1.8, label=f'Threshold Financeiro ({THRESH_FIN:.3f})')
ax1.axhline(0, color='black', lw=0.8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax1.set_ylabel('R$')
ax1.set_title('Impacto Financeiro por Threshold', fontweight='bold')
ax1.legend(fontsize=8, ncol=2)

# Painel 2: recall vs precisão
ax2 = axes[1]
ax2.plot(df_thr['threshold'], df_thr['recall'],   color='#4CAF50', lw=2, label='Recall')
ax2.plot(df_thr['threshold'], df_thr['precisao'], color='#9C27B0', lw=2, label='Precisão')
ax2.axvline(THRESH_OPT, color='gray',    ls='--', lw=1.2)
ax2.axvline(THRESH_FIN, color='#1565C0', ls='-',  lw=1.8)
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Taxa')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.legend(fontsize=9)

fig.suptitle('Tradeoff Financeiro vs Threshold de Classificação', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Tradeoff: Fraudes Detectadas × Transações Legítimas Negadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Painel A: curva TP vs FP (volume absoluto)
ax = axes[0]
ax.plot(df_thr['fp'], df_thr['tp'], color='#2196F3', lw=2.5)

row_f2  = df_thr.iloc[(df_thr['threshold'] - THRESH_OPT).abs().argsort()[:1]]
row_fin = df_thr.iloc[(df_thr['threshold'] - THRESH_FIN).abs().argsort()[:1]]
ax.scatter(row_f2['fp'],  row_f2['tp'],  s=120, color='gray',    zorder=5, label=f'Threshold F2 ({THRESH_OPT:.3f})')
ax.scatter(row_fin['fp'], row_fin['tp'], s=120, color='#F44336', zorder=5, label=f'Threshold Financeiro ({THRESH_FIN:.3f})')
ax.set_xlabel('Transações Legítimas Negadas (FP)')
ax.set_ylabel('Fraudes Detectadas (TP)')
ax.set_title('Volume: Fraudes Detectadas vs Legítimas Negadas', fontweight='bold')
ax.legend(fontsize=9)

# Painel B: razão TP/FP ao longo dos thresholds
ax2 = axes[1]
ratio = df_thr['tp'] / df_thr['fp'].replace(0, np.nan)
ax2.plot(df_thr['threshold'], ratio, color='#4CAF50', lw=2.5)
ax2.axvline(THRESH_OPT, color='gray',    ls='--', lw=1.2, label=f'Threshold F2 ({THRESH_OPT:.3f})')
ax2.axvline(THRESH_FIN, color='#1565C0', ls='-',  lw=1.8, label=f'Threshold Financeiro ({THRESH_FIN:.3f})')
ax2.axhline(1, color='#F44336', ls=':', lw=1.2, label='Breakeven (1 fraude : 1 legítima)')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Fraudes detectadas por transação negada (TP/FP)')
ax2.set_title('Eficiência: Razão TP/FP por Threshold', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(bottom=0)

plt.suptitle('Tradeoff Operacional', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

v_f2  = row_f2.iloc[0]
v_fin = row_fin.iloc[0]
print(f'Threshold F2   → {v_f2["tp"]:.0f} fraudes detectadas, {v_f2["fp"]:.0f} legítimas negadas  (razão {v_f2["tp"]/max(v_f2["fp"],1):.1f}:1)')
print(f'Threshold Fin. → {v_fin["tp"]:.0f} fraudes detectadas, {v_fin["fp"]:.0f} legítimas negadas (razão {v_fin["tp"]/max(v_fin["fp"],1):.1f}:1)')

---
## 7. Comparação: Com Modelo vs Sem Modelo

Baseline **sem modelo**: todas as transações são aprovadas → 100% das fraudes passam.

In [ ]:
total_fraudes = int(y_arr.sum())
total_legit   = int((y_arr == 0).sum())
total_tx      = len(y_arr)

# Com modelo — threshold F2
p_f2 = (prob_val >= THRESH_OPT).astype(int)
tn_f2, fp_f2, fn_f2, tp_f2 = confusion_matrix(y_arr, p_f2).ravel()
salvo_f2  = amount_val[(p_f2==1) & (y_arr==1)].sum()
custo_f2  = fp_f2 * CUSTO_FP
resid_f2  = amount_val[(p_f2==0) & (y_arr==1)].sum()
benef_f2  = salvo_f2 - custo_f2

# Com modelo — threshold financeiro
p_fin = (prob_val >= THRESH_FIN).astype(int)
tn_fin, fp_fin, fn_fin, tp_fin = confusion_matrix(y_arr, p_fin).ravel()
salvo_fin = amount_val[(p_fin==1) & (y_arr==1)].sum()
custo_fin = fp_fin * CUSTO_FP
resid_fin = amount_val[(p_fin==0) & (y_arr==1)].sum()
benef_fin = salvo_fin - custo_fin

cenarios = {
    'Sem Modelo': {
        'Fraudes Detectadas': 0, 'FP': 0,
        'Valor Salvo (R$)': 0, 'Custo FP (R$)': 0,
        'Exposição Residual (R$)': perda_sem_modelo,
        'Benefício Líquido (R$)': -perda_sem_modelo,
    },
    f'Modelo (thr={THRESH_OPT:.3f})': {
        'Fraudes Detectadas': tp_f2, 'FP': fp_f2,
        'Valor Salvo (R$)': salvo_f2, 'Custo FP (R$)': custo_f2,
        'Exposição Residual (R$)': resid_f2,
        'Benefício Líquido (R$)': benef_f2,
    },
    f'Modelo (thr={THRESH_FIN:.3f}) [Financeiro]': {
        'Fraudes Detectadas': tp_fin, 'FP': fp_fin,
        'Valor Salvo (R$)': salvo_fin, 'Custo FP (R$)': custo_fin,
        'Exposição Residual (R$)': resid_fin,
        'Benefício Líquido (R$)': benef_fin,
    },
}

df_cen = pd.DataFrame(cenarios).T
display(
    df_cen.style
    .format({
        'Fraudes Detectadas': '{:.0f}', 'FP': '{:.0f}',
        'Valor Salvo (R$)': 'R$ {:,.2f}', 'Custo FP (R$)': 'R$ {:,.2f}',
        'Exposição Residual (R$)': 'R$ {:,.2f}', 'Benefício Líquido (R$)': 'R$ {:,.2f}',
    })
    .background_gradient(subset=['Benefício Líquido (R$)'], cmap='RdYlGn')
    .set_caption('Comparação de Cenários — Conjunto de Validação')
)

print(f'\nO modelo (thr={THRESH_OPT:.3f}) evita R$ {benef_f2:,.2f} de perda líquida')
print(f'Redução de {benef_f2/perda_sem_modelo:.2%} da exposição total vs sem modelo')

---
## 8. Impacto Financeiro por Safra (Temporal)

In [ ]:
df_safra = pd.DataFrame({
    'safra' : safra_val,
    'target': y_arr,
    'score' : prob_val,
    'pred'  : p_f2,
    'amount': amount_val,
})

safras_ord = sorted(df_safra['safra'].unique())
rows = []
for s in safras_ord:
    sub = df_safra[df_safra['safra'] == s]
    tn_, fp_, fn_, tp_ = confusion_matrix(sub['target'], sub['pred']).ravel()
    salvo_ = sub.loc[(sub['pred']==1) & (sub['target']==1), 'amount'].sum()
    resid_ = sub.loc[(sub['pred']==0) & (sub['target']==1), 'amount'].sum()
    custo_ = fp_ * CUSTO_FP
    rows.append({
        'Safra': s, 'Total TX': len(sub), 'Fraudes': int(sub['target'].sum()),
        'Taxa Fraude': sub['target'].mean(),
        'TP': tp_, 'FP': fp_, 'FN': fn_,
        'Recall': tp_ / (tp_ + fn_) if tp_ + fn_ > 0 else 0,
        'Valor Salvo': salvo_, 'Custo FP': custo_, 'Exposição FN': resid_,
        'Benefício Líq.': salvo_ - custo_,
    })

df_s = pd.DataFrame(rows)
display(
    df_s.style
    .format({
        'Taxa Fraude': '{:.2%}', 'Recall': '{:.2%}',
        'Valor Salvo': 'R$ {:,.2f}', 'Custo FP': 'R$ {:,.2f}',
        'Exposição FN': 'R$ {:,.2f}', 'Benefício Líq.': 'R$ {:,.2f}',
    })
    .background_gradient(subset=['Benefício Líq.'], cmap='RdYlGn')
    .set_caption(f'Impacto Financeiro por Safra — threshold {THRESH_OPT:.3f}')
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})
x = np.arange(len(safras_ord))

ax1 = axes[0]
ax1.bar(x, df_s['Valor Salvo'],   color='#4CAF50', alpha=0.85, label='Valor Salvo (TP)')
ax1.bar(x, -df_s['Custo FP'],     color='#F44336', alpha=0.75, label='Custo FP')
ax1.bar(x, -df_s['Exposição FN'], color='#FF9800', alpha=0.60, label='Exposição Residual (FN)')
ax1.plot(x, df_s['Benefício Líq.'], color='#1565C0', marker='o', lw=2.5, ms=8, label='Benefício Líq.', zorder=5)
ax1.axhline(0, color='black', lw=0.8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v:,.0f}'))
ax1.set_ylabel('R$')
ax1.set_title('Impacto Financeiro por Safra', fontweight='bold')
ax1.legend(fontsize=9, ncol=2)

ax2 = axes[1]
ax2.plot(x, df_s['Recall'], color='#4CAF50', marker='o', lw=2, ms=7)
ax2.fill_between(x, df_s['Recall'], alpha=0.15, color='#4CAF50')
ax2.axhline(df_s['Recall'].mean(), color='gray', ls='--', lw=1.2, label=f"Média {df_s['Recall'].mean():.2%}")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax2.set_xticks(x)
ax2.set_xticklabels(safras_ord, rotation=20, ha='right', fontsize=10)
ax2.set_ylabel('Recall')
ax2.set_title('Recall de Fraude por Safra', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Estabilidade Financeira Temporal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Exposição Residual — Fraudes que Ainda Passam (FN)

Análise das fraudes não detectadas: quanto valem, qual a concentração de valor.

In [ ]:
fn_mask_f2 = (p_f2 == 0) & (y_arr == 1)
amount_fn  = amount_val[fn_mask_f2]

print(f'Fraudes não detectadas (FN) : {fn_f2:,}')
print(f'Valor total FN              : R$ {amount_fn.sum():,.2f}')
print(f'Valor médio por FN          : R$ {amount_fn.mean():.2f}')
print(f'FN de alto valor (> R$100)  : {(amount_fn > 100).sum()} transações  ->  R$ {amount_fn[amount_fn > 100].sum():,.2f}')
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma de valor dos FN
ax1 = axes[0]
ax1.hist(amount_fn, bins=40, color='#FF7043', edgecolor='white', alpha=0.85)
ax1.axvline(amount_fn.mean(), color='black', ls='--', lw=1.5, label=f'Média R${amount_fn.mean():.2f}')
ax1.set_xlabel('Valor da Transação (R$)')
ax1.set_ylabel('Frequência')
ax1.set_title('Distribuição de Valor — Fraudes Não Detectadas (FN)', fontweight='bold')
ax1.legend(fontsize=9)

# Curva de Pareto (concentração de valor)
sorted_fn  = np.sort(amount_fn)[::-1]
cumsum_fn  = np.cumsum(sorted_fn) / sorted_fn.sum()
pct_casos  = np.arange(1, len(sorted_fn) + 1) / len(sorted_fn)

ax2 = axes[1]
ax2.plot(pct_casos * 100, cumsum_fn * 100, color='#FF7043', lw=2.5)
ax2.axhline(80, color='gray', ls=':', lw=1, label='80% do valor total')
idx_80 = np.searchsorted(cumsum_fn, 0.80)
if idx_80 < len(pct_casos):
    ax2.axvline(pct_casos[idx_80] * 100, color='gray', ls=':', lw=1)
    ax2.annotate(f'{pct_casos[idx_80]:.1%} dos FN',
                 xy=(pct_casos[idx_80]*100, 80),
                 xytext=(pct_casos[idx_80]*100 + 5, 60),
                 fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))
ax2.set_xlabel('% das Fraudes Não Detectadas (ordenadas por valor desc.)')
ax2.set_ylabel('% do Valor Total Acumulado')
ax2.set_title('Concentração de Valor — Fraudes Não Detectadas (Pareto)', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 10. Análise de Sensibilidade

Como o benefício líquido se comporta sob diferentes premissas de custo? Isso testa a robustez da decisão de implantar o modelo.

In [ ]:
custos_revisao = [1, 2, 5, 10, 20, 50]
taxas_churn    = [0.00, 0.01, 0.02, 0.05, 0.10]

grid = pd.DataFrame(
    index  =[f'R${c:.0f}/FP revisao' for c in custos_revisao],
    columns=[f'{t:.0%} churn' for t in taxas_churn],
    dtype=float
)

for cr in custos_revisao:
    for tc in taxas_churn:
        custo_fp_i = cr + tc * LTV_CLIENTE
        benef_i    = salvo_f2 - fp_f2 * custo_fp_i
        grid.loc[f'R${cr:.0f}/FP revisao', f'{tc:.0%} churn'] = benef_i

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    grid.astype(float), annot=True, fmt=',.0f', cmap='RdYlGn',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Benefício Líquido (R$)'},
)
ax.set_xlabel('Taxa de Churn por FP')
ax.set_ylabel('Custo Operacional de Revisão')
ax.set_title(
    f'Sensibilidade do Benefício Líquido (thr={THRESH_OPT:.3f})\n'
    f'LTV fixo = R${LTV_CLIENTE:.0f} | TP = {tp_f2} fraudes bloqueadas | FP = {fp_f2} legítimas negadas',
    fontweight='bold'
)
plt.tight_layout()
plt.show()

print(f'Pior cenário  (R$50/FP, 10% churn) : R$ {grid.iloc[-1, -1]:,.2f}')
print(f'Melhor cenário (R$1/FP,  0% churn) : R$ {grid.iloc[ 0,  0]:,.2f}')
print(f'Cenário base   (R${CUSTO_REVISAO_FP:.0f}/FP, {TAXA_CHURN_FP:.0%} churn) : R$ {benef_f2:,.2f}')

---
## 11. Sumário Executivo

In [ ]:
roi_vs_nada = benef_f2 / perda_sem_modelo

print('=' * 62)
print(' SUMÁRIO EXECUTIVO — DETECCAO DE FRAUDE')
print('=' * 62)
print()
print(f'  Período analisado     : {safras_ord[0]} a {safras_ord[-1]}')
print(f'  Transações avaliadas  : {total_tx:,}')
print(f'  Fraudes reais         : {total_fraudes:,}  ({total_fraudes/total_tx:.2%} do volume)')
print()
print('  DESEMPENHO DO MODELO')
print(f'    Threshold aplicado  : {THRESH_OPT:.3f}')
print(f'    Fraudes bloqueadas  : {tp_f2:,} de {total_fraudes:,}  ({tp_f2/total_fraudes:.2%} recall)')
print(f'    Legítimas negadas   : {fp_f2:,} de {total_legit:,}  ({fp_f2/total_legit:.4%} do volume legítimo)')
print(f'    Razão TP:FP         : {tp_f2/max(fp_f2,1):.1f} fraudes bloqueadas por cada legítima negada')
print()
print('  IMPACTO FINANCEIRO (conjunto de validação)')
print(f'    Fraudes evitadas (R$)       : R$ {salvo_f2:>10,.2f}')
print(f'    Custo das negações (R$)     : R$ {custo_f2:>10,.2f}')
print(f'    Exposição residual (FN)     : R$ {resid_f2:>10,.2f}')
print(f'    Benefício líquido           : R$ {benef_f2:>10,.2f}')
print()
print(f'    Reducao da exposição total  : {roi_vs_nada:.2%}  (vs cenário sem modelo)')
print()
print('  THRESHOLD RECOMENDADO')
if abs(THRESH_FIN - THRESH_OPT) > 0.01:
    diff = BEN_FIN - BEN_F2
    print(f'    Threshold F2 (atual)        : {THRESH_OPT:.3f}')
    print(f'    Threshold financeiro ótimo  : {THRESH_FIN:.3f}')
    print(f'    Ganho adicional potencial   : R$ {diff:,.2f}')
    print(f'    Recomendacao: considerar ajuste para {THRESH_FIN:.3f} para maximizar ROI')
else:
    print(f'    O threshold F2 ({THRESH_OPT:.3f}) coincide com o otimo financeiro.')
print('=' * 62)